In [3]:
import json

import  numpy
import  os
from openai import OpenAI

In [4]:
model="gpt-4.1mini"

In [ ]:
client=OpenAI(
    api_key="your api key",
)


In [ ]:
prompt="write a one sentence bedtime  story about a unicorn"

response=client.responses.create(
    model=model,
    input=prompt,

)
print('prompt:', prompt)
print('response:', response.output_text)

print("\n with controlled parameters:")

response=client.responses.create(
    model=model,
    input=prompt,
    temperature=0.7,
    top_p=0.9,
)

print(f"response{response.output_text}")

In [ ]:
response=client.responses.create(
    model=model,
    input=[{"role":"system","content":"you a id generator ai.convert the user input into a UI"},{"role":"user","content":"make a user profile from"}]
,  text={
        "format": {
            "type": "json_schema",
            "name": "ui",
            "description": "Dynamically generated UI",
            "schema": {
                "type": "object",
                "properties": {
                    "type": {
                        "type": "string",
                        "description": "The type of the UI component",
                        "enum": ["div", "button", "header", "section", "field", "form"]
                    },
                    "label": {
                        "type": "string",
                        "description": "The label of the UI component, used for buttons or form fields"
                    },
                    "children": {
                        "type": "array",
                        "description": "Nested UI components",
                        "items": {"$ref": "#"}
                    },
                    "attributes": {
                        "type": "array",
                        "description": "Arbitrary attributes for the UI component, suitable for any element",
                        "items": {
                            "type": "object",
                            "properties": {
                              "name": {
                                  "type": "string",
                                  "description": "The name of the attribute, for example onClick or className"
                              },
                              "value": {
                                  "type": "string",
                                  "description": "The value of the attribute"
                              }
                          },
                          "required": ["name", "value"],
                          "additionalProperties": False
                      }
                    }
                },
                "required": ["type", "label", "children", "attributes"],
                "additionalProperties": False
            },
            "strict": True,
        },
    },
)
ui=json.loads(response.output_text)

In [ ]:
from pydantic import BaseModel

class Step(BaseModel):
    explanation: str
    output: str

class MathReasoning(BaseModel):
    steps: list[Step]
    final_answer: str

completion = client.beta.chat.completions.parse(
    model=model,
    messages=[
        {"role": "system", "content": "You are a helpful math tutor. Guide the user through the solution step by step."},
        {"role": "user", "content": "how can I solve 8x + 7 = -23"}
    ],
    response_format=MathReasoning,
)

math_reasoning = completion.choices[0].message

if(math_reasoning.refusal):
    print(math_reasoning.refusal)
else:
    print(math_reasoning.parsed)

In [ ]:
image_url="https://images.unsplash.com/photo-1579546929518-9e396f3cc809?ixlib=rb-4.0.3&ixid=MnwxMjA3fDB8MHxleHBsb3JlLWZlZWR8MXx8fGVufDB8fHx8&w=1000&q=8"

response=client.responses.create(
    model=model,
    input=[{
        "role": "user",
        "content": [
            {"type": "input_text", "text": "what's in this image?"},
            {
                "type": "input_image",
                "image_url": image_url,
            },
        ],
    }],
)

print(response.output_text)

In [ ]:
speech_file="speech.mp3"

with client.audio.speech.with_streaming_response.create(
    model="gpt-4o-mini-tts",
    voice="coral",
    input="today is a wonderful day to do build something people love",
    instructions="speck in a cheerful and  positive tone.",
)as response:
    response.stream_to_file(speech_file)



In [ ]:
audio_file=open("speech.mp3","rb")
transcription=client.audio.transcriptions.create(
    model="gpt-4o-transcribe",
    file=audio_file,
)
print(transcription.text)

In [ ]:
import requests

def get_weather(latitude, longitude):
    response=requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m")

    return data['current']['temperature_2m']
tools = [{
    "type": "function",
    "name": "get_weather",
    "description": "Get current temperature for provided coordinates in celsius.",
    "parameters": {
        "type": "object",
        "properties": {
            "latitude": {"type": "number"},
            "longitude": {"type": "number"}
        },
        "required": ["latitude", "longitude"],
        "additionalProperties": False
    },
    "strict": True
}]

input_messages=[{'role':'user','content':'what is the weather today'}]

response=client.responses.create(
    model="gpt-4.1-mini",
    input=input_messages,
    tools=tools,
)


In [ ]:
tool_call=response.output[0]
args=json.loads(tool_call.arguments)

result=get_weather(args['latitude'],args['longitude'])

In [ ]:
response.output[0].arguments

In [ ]:
input_messages.append(tool_call)
input_messages.append({"type":"function","call_id":tool_call.call_id,
                       "output":str(result)})

response_2=client.responses.create(
    model="gpt-4.1-mini",
    input=input_messages,
    tools=tools,
)
print(response_2.output_text)

In [ ]:
response_2=client.responses.create(
    model="gpt-o4-mini",
    input="explain how to implement a hash table",
)
print("\n o3-mini response:")
print(response_2.output_text)

In [ ]:
response_o4=client.responses.create(
    model="o4-mini",
    input="explain how to implement a hash table",


)
print("\no3-mini response:")
print(response_o4.output_text)

In [ ]:
import numpy as np
words = [
    "king", "queen", "man", "woman",
    "apple", "banana", "orange", "pear",
    "castle", "throne"
]

response=client.embeddings.create(
    model="text-embedding-3-large",
    input=words,
    encoding_format="float"
)

embeddings=[data.embedding for data in response.output]

print(f"Embeddings dimennsion:{len({embeddings})}")
print(f"number of words:{len(embeddings)}")


def dot_product(vec1, vec2):
    return np.dot(vec1, vec2)

similarity_matrix = np.zeros((len(words), len(words)))
for i in range(len(words)):
    for j in range(len(words)):
        similarity_matrix[i][j] = dot_product(embeddings[i], embeddings[j])

print("\nSimilarity Matrix (Dot Products):")
print("          " + " ".join(f"{word:<8}" for word in words))
for i, word in enumerate(words):
    row_values = " ".join(f"{similarity_matrix[i][j]:.4f}  " for j in range(len(words)))
    print(f"{word:<10} {row_values}")

king_queen_similarity = dot_product(embeddings[0], embeddings[1])
apple_banana_similarity = dot_product(embeddings[4], embeddings[5])

print("\nSpecific relationships:")
print(f"Similarity between 'king' and 'queen': {king_queen_similarity:.4f}")
print(f"Similarity between 'apple' and 'banana': {apple_banana_similarity:.4f}")
print(f"Similarity between 'king' and 'apple': {dot_product(embeddings[0], embeddings[4]):.4f}")